# Entrenamiento remoto con GPU en Colab

Este notebook prepara el entrenamiento fuera de la GPU local. Puede trabajar de dos formas:

- `SOURCE_MODE = "git"`: clona el repositorio desde GitHub. Usalo cuando la rama `Victus-rama` ya este subida.
- `SOURCE_MODE = "upload_zip"`: sube un ZIP liviano generado con `scripts/package-colab-project.ps1`. Usalo cuando quieras entrenar con cambios locales aun no subidos a GitHub.

Antes de ejecutar: confirma que el kernel sea Colab y que el runtime tenga GPU.


In [ ]:
# Configuracion principal
SOURCE_MODE = "git"  # "upload_zip" o "git"
REPO_URL = "https://github.com/AbdyHernandez-proy/clasificador-imagenes.git"
BRANCH = "Victus-rama"
PROJECT_DIR = "/content/clasificador-imagenes"

# Presets disponibles: yolo_coco128, faster_voc_short, retinanet_voc_short
SELECTED_PRESET = "yolo_coco128"

PRESETS = {
    "yolo_coco128": {
        "models": "yolo",
        "dataset": "coco128-detect",
        "epochs": 25,
        "batch": 8,
        "image_size": 640,
        "session_samples": 0,
        "session_count": 1,
        "log_every": 25,
        "publish_partial": False,
    },
    "faster_voc_short": {
        "models": "faster-rcnn",
        "dataset": "voc-detect",
        "epochs": 3,
        "batch": 2,
        "image_size": 512,
        "session_samples": 2048,
        "session_count": 1,
        "log_every": 25,
        "publish_partial": True,
    },
    "retinanet_voc_short": {
        "models": "retinanet",
        "dataset": "voc-detect",
        "epochs": 3,
        "batch": 2,
        "image_size": 512,
        "session_samples": 2048,
        "session_count": 1,
        "log_every": 25,
        "publish_partial": True,
    },
}

USE_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/clasificador-imagenes-colab"


In [ ]:
# Verificar GPU remota
import os
import subprocess
import sys
from pathlib import Path

try:
    import torch
except ImportError:
    torch = None

if torch is None or not torch.cuda.is_available():
    raise RuntimeError("No hay GPU CUDA disponible. En Colab usa Runtime/Entorno de ejecucion > Cambiar tipo > GPU.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("Python:", sys.version)


In [ ]:
# Cargar codigo del proyecto en /content
from pathlib import Path
import os
import shutil
import subprocess
import zipfile

project_dir = Path(PROJECT_DIR)

if SOURCE_MODE == "git":
    if project_dir.exists():
        shutil.rmtree(project_dir)
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(project_dir)], check=True)
elif SOURCE_MODE == "upload_zip":
    from google.colab import files
    print("Sube el ZIP generado con: scripts/package-colab-project.ps1")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No se subio ningun ZIP.")
    zip_name = next(iter(uploaded.keys()))
    if project_dir.exists():
        shutil.rmtree(project_dir)
    project_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(project_dir)
else:
    raise ValueError(f"SOURCE_MODE no soportado: {SOURCE_MODE}")

os.chdir(project_dir)
print("Proyecto listo en:", project_dir)
print(subprocess.check_output(["python", "-c", "import pathlib; print(pathlib.Path.cwd())"], text=True))


In [ ]:
# Montar Google Drive para guardar resultados persistentes
from pathlib import Path
import shutil

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    print("Salida persistente:", DRIVE_OUTPUT_DIR)
else:
    print("Drive desactivado. Los resultados quedaran solo en el runtime temporal.")


In [ ]:
# Instalar dependencias de entrenamiento
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pillow", "PyYAML", "numpy", "ultralytics"], check=True)

# Colab normalmente ya trae torch/torchvision con CUDA. Solo se valida aqui.
import torch
import torchvision
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda disponible:", torch.cuda.is_available())


In [ ]:
# Preparar dataset segun preset
import subprocess
import sys

preset = PRESETS[SELECTED_PRESET]
dataset_id = preset["dataset"]

if dataset_id == "voc-detect":
    subprocess.run([sys.executable, "-m", "ml.training.prepare_voc"], check=True)
elif dataset_id in {"coco8-detect", "coco128-detect"}:
    subprocess.run([sys.executable, "backend/manage_datasets.py", "download", dataset_id], check=True)
else:
    raise ValueError(f"Dataset no preparado automaticamente por este notebook: {dataset_id}")

print("Dataset preparado:", dataset_id)


In [ ]:
# Entrenar en GPU remota
import subprocess
import sys

preset = PRESETS[SELECTED_PRESET]
cmd = [
    sys.executable,
    "-m", "ml.training.local_train",
    "--models", preset["models"],
    "--dataset-id", preset["dataset"],
    "--epochs", str(preset["epochs"]),
    "--batch", str(preset["batch"]),
    "--imgsz", str(preset["image_size"]),
    "--device", "0",
    "--session-samples", str(preset["session_samples"]),
    "--session-count", str(preset["session_count"]),
    "--log-every", str(preset["log_every"]),
    "--workers", "2",
    "--resume",
]
if preset.get("publish_partial"):
    cmd.append("--publish-partial")

print("Ejecutando:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Guardar artefactos y registros en Google Drive
from pathlib import Path
import shutil
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
export_root = Path(DRIVE_OUTPUT_DIR) / f"run-{SELECTED_PRESET}-{stamp}" if USE_DRIVE else Path("/content") / f"run-{SELECTED_PRESET}-{stamp}"
export_root.mkdir(parents=True, exist_ok=True)

items = [
    "ml/models",
    "ml/registry.json",
    "ml/datasets/registry.json",
    "docs/REGISTRO_ENTRENAMIENTOS.md",
]
for item in items:
    src = Path(item)
    if not src.exists():
        continue
    dst = export_root / item
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

archive_path = shutil.make_archive(str(export_root), "zip", export_root)
print("Artefactos guardados en:", export_root)
print("ZIP:", archive_path)


## Como traer resultados de vuelta

Cuando termine el entrenamiento, descarga desde Drive el ZIP generado en `clasificador-imagenes-colab/run-...zip`.

Los artefactos importantes estaran dentro de:

- `ml/models/`
- `ml/registry.json`
- `ml/datasets/registry.json`

Luego me avisas y reviso/integraré esos resultados en el proyecto local.
